<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_31_Linux_SSH_Auth_Log_Suspicious_Login_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# Experiment 3: Linux SSH Auth Log Suspicious Login Detector
# ==========================================================

import re

# Regular expression to parse SSH authentication log entries
AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) "
    r"from (?P<ip>[\d.]+) port (?P<port>\d+)"
)


# ----------------------------------------------------------
# Function to parse auth.log entries
# ----------------------------------------------------------
def parse_auth_log(lines):
    """
    Parse Linux SSH authentication log lines into
    structured dictionaries.
    """

    entries = []

    for line in lines:

        match = AUTH_LINE_RE.search(line)

        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line
            })

    return entries


# ----------------------------------------------------------
# Function to detect suspicious logins
# ----------------------------------------------------------
def flag_suspicious_logins(entries, trusted_ips):
    """
    Flag successful SSH logins from untrusted IP addresses.
    Root logins are marked HIGH severity.
    """

    flagged = []

    for entry in entries:

        if (
            entry["result"] == "Accepted"
            and entry["ip"] not in trusted_ips
        ):

            severity = (
                "HIGH"
                if entry["user"] == "root"
                else "MEDIUM"
            )

            flagged.append({
                **entry,
                "severity": severity
            })

    return flagged


# ==========================================================
# Test Cases
# ==========================================================

def test_experiment3():

    log_lines = [

        "Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2",

        "Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2",

        "Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2"

    ]

    trusted_ips = {
        "10.0.0.5"
    }

    # Parse log entries
    entries = parse_auth_log(log_lines)

    print("Parsed Log Entries")
    print("-" * 50)

    for entry in entries:
        print(entry)

    print()

    # Detect suspicious logins
    flagged = flag_suspicious_logins(
        entries,
        trusted_ips
    )

    print("Suspicious Login(s)")
    print("-" * 50)

    for item in flagged:
        print(item)

    print()

    # Assertions
    assert len(entries) == 3
    assert len(flagged) == 1
    assert flagged[0]["user"] == "root"
    assert flagged[0]["ip"] == "198.51.100.23"
    assert flagged[0]["severity"] == "HIGH"

    print("All test cases passed.")


# ----------------------------------------------------------
# Run Test
# ----------------------------------------------------------
test_experiment3()

Parsed Log Entries
--------------------------------------------------
{'result': 'Accepted', 'user': 'deploy', 'ip': '10.0.0.5', 'port': 51100, 'raw': 'Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2'}
{'result': 'Failed', 'user': 'root', 'ip': '198.51.100.23', 'port': 51320, 'raw': 'Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2'}
{'result': 'Accepted', 'user': 'root', 'ip': '198.51.100.23', 'port': 51322, 'raw': 'Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2'}

Suspicious Login(s)
--------------------------------------------------
{'result': 'Accepted', 'user': 'root', 'ip': '198.51.100.23', 'port': 51322, 'raw': 'Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2', 'severity': 'HIGH'}

All test cases passed.
